# 01 — Visualização de Séries Temporais Estratificadas por LCZ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/01_lcz_time_series.pt.ipynb)

**Objetivo de aprendizagem**: aprender como o LCZ4py atribui a cada estação meteorológica uma classe de Zona Climática Local (LCZ) e usa essa classe como variável de agrupamento para visualizar séries temporais de temperatura do ar — o passo fundamental para toda análise climática urbana baseada em estação ("local") neste pacote, da detecção de anomalias à intensidade da Ilha de Calor Urbana (ICU).

> **Nota de renderização:** todas as figuras deste caderno são geradas por padrão com o renderizador estático nativo do LCZ4py (matplotlib, exportação em 300 dpi PNG/PDF/SVG/TIFF, sem precisar de Plotly/Kaleido). Passe `renderer="plotly"` para uma função de plotagem para obter a figura interativa do Plotly.

## Por que séries temporais estratificadas por LCZ?

O esquema de Zonas Climáticas Locais (Stewart & Oke, 2012) classifica paisagens urbanas e naturais em 17 categorias padronizadas (LCZ 1–10: tipos construídos, de compacto alto a esparsamente construído; LCZ 11(A)–17(G): tipos de cobertura do solo, de floresta densa a água) com base em propriedades físicas que controlam o balanço de energia superficial local — fator de visão do céu, fração de superfície construída/impermeável, admitância superficial e emissão de calor antropogênico. Duas estações a 500 m de distância podem estar em LCZs muito diferentes (por exemplo, um pátio compacto de médio porte vs. um parque) e, portanto, apresentar regimes térmicos sistematicamente distintos, mesmo que uma rede convencional "urbana vs. rural" as agrupasse ou, pior, descartasse a comparação por falta de um referencial comum.

**A família de módulos `local/` do LCZ4py existe para tornar essa estratificação automática.** Dado um raster de LCZ classificado (por exemplo, de `lcz_get_map`) e uma tabela de observações de estações com latitude/longitude, `lcz_ts` realiza uma junção espacial ponto-em-raster: cada estação recebe a classe LCZ do pixel em que está localizada. A partir daí, cada estação carrega um rótulo LCZ, e as séries temporais podem ser agrupadas, comparadas e organizadas em painéis por categoria de forma urbana, em vez de tratadas como um conjunto indiferenciado de sensores. Esse é exatamente o fluxo de trabalho descrito para o pacote R no qual esta versão Python se baseia — Anjos, M. et al. (2025), *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0 — e é o pré-requisito para toda função `local/` subsequente: anomalias de temperatura (caderno 02), intensidade da ICU (03), interpolação espacial (04–05) e métricas climáticas derivadas (06–07).

**A agregação temporal (`time_freq`)** controla a resolução da janela de média antes da plotagem:
- `"hour"` — preserva o ciclo diurno; ideal para observar contrastes dia/noite (por exemplo, ICU noturna).
- `"day"` — suaviza o ruído horário, útil para visões gerais de vários meses ou anos.
- `"month"` — revela o ciclo sazonal, útil para comparações ano a ano.

**`facet_plot='lcz'`** (usado com `plot_type="heatmap"`) coloca uma linha por classe LCZ em um eixo temporal compartilhado, tornando visualmente óbvio se, digamos, as classes urbanas compactas (LCZ 2–3) ficam sistematicamente mais quentes à noite do que as classes abertas/naturais (LCZ 11–14) — a assinatura diurna característica da ilha de calor urbana. O parâmetro `by=` oferece uma decomposição complementar — dividindo em subgráficos por `"season"`, `"year"`, `"weekday"`/`"weekend"`, ou um modo especial `"daylight"` que sombreia as janelas diurnas diretamente em um único gráfico unificado.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Montando o pipeline de dados compartilhado: mapa LCZ de Berlim + observações de estações

Este é o primeiro caderno da série `local/`, portanto ele monta os dois artefatos que todo caderno `local/` posterior reutiliza:
1. **`lcz_map`** — o GeoTIFF de LCZ classificado para Berlim, baixado uma vez com `lcz_get_map` (mapa global, Demuzere et al. 2022) e armazenado em cache localmente.
2. **`df`** — observações horárias de temperatura do ar de 23 estações na região de Berlim (`lcz_data_berlin.csv`, 2019–2020), carregadas diretamente da URL bruta do GitHub para funcionar em qualquer sessão nova do Colab sem upload.

Colunas em `lcz_data_berlin.csv`: `date` (data/hora), `station` (id da estação), `airT` (temperatura do ar, °C), `Latitude`, `Longitude`.

In [ ]:
import pandas as pd
from IPython.display import display
from LCZ4py import lcz_get_map
from LCZ4py.local.lcz_ts import lcz_ts

# 1a. Baixar (ou carregar do cache) o mapa LCZ de Berlim
lcz_map = lcz_get_map(city="Berlin")
print("Mapa LCZ:", lcz_map)

# 1b. Carregar o conjunto de dados de referência usado em todos os cadernos local/
df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
print(df.shape)
df.head()

`lcz_map` é o caminho para um GeoTIFF recortado (classes 1–17) cobrindo a extensão de Berlim. `df` tem uma linha por estação-hora. Nenhum dos dois objetos ainda codifica a associação LCZ — isso acontece dentro da própria `lcz_ts`, via junção espacial contra `lcz_map`, usando `Latitude`/`Longitude` para localizar a classe do pixel de cada estação.

## 2. Visão padrão: `plot_type="basic_line"`

A chamada mais simples agrupa por estação, calcula a média na `time_freq` solicitada e desenha uma linha por estação. `lcz_ts` anexa a classe LCZ resolvida ao nome de cada estação na legenda (por exemplo, `tiergarten (14)`), então já é possível ver o agrupamento por forma urbana sem qualquer painel adicional. Aqui agregamos em médias diárias ao longo de todo o registro para manter o gráfico legível.

## Exportação pronta para publicação com `style`

As funções de plotagem agora aceitam `style="default"`, `style="nature"`, `style="science"` e `style="generic_bw"`. Os presets de revista ajustam a família de fontes, o tamanho físico em milímetros, o DPI (300 para os estilos voltados à publicação) e as cores, de modo que o mesmo gráfico de `lcz_ts` possa ser exportado como figura pronta para manuscrito com um argumento extra.

Um exemplo mínimo pronto para publicação:

- `style="science"` para a figura de série temporal.
- `isave=True` com `save_extension="png"` ou `"pdf"` para gravar a figura final.


In [ ]:
fig_line_pub = lcz_ts(
    lcz_map,
    data_frame=df,
    var="airT",
    station_id="station",
    time_freq="day",
    plot_type="basic_line",
    style="science",
    isave=True,
    save_extension="png",
    lang="pt",
)
display(fig_line_pub)


In [ ]:
fig_line = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="day", plot_type="basic_line", lang="pt",
    title="Temperatura do ar nas estações de Berlim — média diária, por estação rotulada com LCZ",
)
display(fig_line)

**Interpretando este gráfico**: cada traço é uma estação, rotulada com sua classe LCZ entre parênteses. As linhas se agrupam em torno do ciclo sazonal compartilhado (frio de dez–fev, quente de jun–ago), mas em qualquer data a *dispersão* entre as linhas é a parte interessante — essa dispersão é em grande parte explicada pela LCZ em que cada estação está, o que os próximos gráficos tornam explícito.

## 3. Painéis sazonais: `by="season"`

Definir `by="season"` divide a figura em um subgráfico por estação do ano, cada um mostrando as mesmas linhas por estação restritas às datas daquela estação do ano. Isso isola se a dispersão de temperatura entre estações, causada pela LCZ, é estável ao longo do ano ou muda sazonalmente (por exemplo, efeitos de ICU costumam ser mais fortes em noites de verão e mais fracos sob cobertura de neve no inverno).

In [ ]:
fig_season = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="day", by="season", lang="pt",
    title="Temperatura do ar nas estações de Berlim por estação do ano",
)
display(fig_season)

**Interpretando este gráfico**: compare a dispersão vertical das linhas de estação dentro de cada subgráfico sazonal. Uma dispersão maior no verão do que no inverno é a assinatura sazonal clássica do contraste térmico urbano — LCZs construídas compactas retêm e reirradiam calor solar, enquanto LCZs abertas/naturais resfriam com mais eficiência, e essa diferença se amplia sob forte insolação de verão.

## 4. Mapa de calor com painéis por LCZ: `plot_type="heatmap"`, `facet_plot="lcz"`

A visão em mapa de calor é a forma mais direta de ver a assinatura diurna de cada classe LCZ: uma linha por classe LCZ, hora no eixo x, cor = temperatura média. Aqui restringimos a julho de 2019 (verão do hemisfério norte, quando os contrastes de ICU costumam ser mais nítidos) e agregamos por hora, tornando o ciclo diurno visível.

In [ ]:
fig_heatmap = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="hour", plot_type="heatmap", facet_plot="lcz",
    year=[2019], month=[7], lang="pt",
    title="Temperatura horária do ar em Berlim por classe LCZ — julho de 2019",
)
display(fig_heatmap)

**Interpretando este gráfico**: cada linha é uma classe LCZ presente entre as estações de Berlim. Procure linhas que permanecem quentes (cor mais clara) durante as horas noturnas — essas são as classes compactas/urbanas (LCZ 1–3, 5–6) retendo calor — em contraste com linhas que esfriam bruscamente após o pôr do sol, tipicamente classes abertas ou vegetadas (LCZ 11–14). Essa persistência noturna de calor nas classes construídas é a evidência visual direta da ilha de calor urbana de dossel que os próximos cadernos (02, 03) irão quantificar numericamente.

## 5. Visão com sombreamento diurno: `by="daylight"`

`by="daylight"` mantém todas as estações em um único eixo temporal unificado (como o gráfico de linha básico), mas sombreia as janelas diurnas em âmbar, calculadas por data a partir da fórmula de elevação solar da NOAA usando a latitude/longitude média das estações. Isso facilita separar visualmente o aquecimento diurno do comportamento noturno sem dividir o gráfico em subgráficos — útil para identificar se a dispersão entre estações é um fenômeno diurno ou noturno.

In [ ]:
fig_daylight = lcz_ts(
    x=lcz_map, data_frame=df, var="airT", station_id="station",
    time_freq="hour", by="daylight",
    year=[2019], month=[7], lang="pt",
    title="Temperatura do ar em Berlim, julho de 2019 — janelas diurnas sombreadas",
)
display(fig_daylight)

**Interpretando este gráfico**: a temperatura sobe durante cada faixa âmbar (diurna) e cai nos intervalos noturnos sem sombreamento. Se a dispersão entre estações se estreita visivelmente durante as faixas diurnas sombreadas e se amplia nos intervalos noturnos, isso é evidência direta de que a ilha de calor urbana em Berlim é predominantemente um fenômeno **noturno** — o padrão clássico para cidades de latitude média, onde LCZs urbanas compactas liberam à noite o calor armazenado durante o dia, enquanto LCZs abertas/rurais irradiam livremente para um céu limpo.

## Conclusão

Este caderno montou o pipeline de dados compartilhado da série `local/` (mapa LCZ de Berlim + `lcz_data_berlin.csv`) e mostrou como `lcz_ts` transforma observações brutas de estações em séries temporais estratificadas por LCZ através de uma única junção espacial, visualizando-as em seguida com visões de linha, painéis sazonais, mapa de calor com painéis por LCZ e sombreamento diurno. Todo caderno `local/` posterior reutiliza esse mesmo par `lcz_map` + `df` para manter a consistência.

**Anterior**: `notebooks/general/07_gridded_climate_environment` (dados climáticos/ambientais em grade — o último caderno da série "general" baseada em raster).

**Próximo**: `notebooks/local/02_temperature_anomalies` — usando as mesmas estações rotuladas por LCZ para calcular anomalias de temperatura por estação em relação à média espacial.